# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya

This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described and accessed via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is available
!pip install -q mlcroissant

## 1. Data Loading

Load dataset metadata and explore its main properties using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
# Convert metadata to dict for inspection/printing purposes
metadata = dataset.metadata.to_json()
print(f"Dataset Name: {metadata.get('name')}")
print(f"Description: {metadata.get('description')}")
print(f"License: {metadata.get('license')}")
print(f"Authors: {[author['@id'] for author in metadata.get('author', [])]}")
print(f"Version: {metadata.get('version')}")
print(f"Published: {metadata.get('datePublished')}")

## 2. Data Overview

List the available record sets and their fields, referencing each by its `@id`.

_Note: If the dataset includes multiple record sets, each will be listed with its `@id` and contained fields._

In [ ]:
# Inspect available record sets and their fields
record_sets = list(dataset.metadata.record_sets)
if not record_sets:
    print("No record sets found in this dataset. Please ensure the Croissant schema includes them.")
else:
    for rs in record_sets:
        print(f"RecordSet Name: {rs.name}")
        print(f"  @id: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id={field.id}, type: {field.data_type})")
        print()

## 3. Data Extraction

Load data from one or more record sets into pandas DataFrames for analysis.

**Note:** Entities (record sets, fields, columns) are always referenced by their `@id`.

In [ ]:
# Gather record set @id values for extraction
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    # Extract all records as a list of dicts
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns: {df.columns.tolist()}")
        display(df.head(3))
    else:
        print(f"No records found for {record_set_id}")

## 4. Exploratory Data Analysis (EDA)

Apply standard EDA steps such as filtering, normalization, and grouping by key fields, referencing all fields by their `@id`.

If you wish to analyze a specific numeric or grouping field, refer to it by its `@id` as shown below.

In [ ]:
# Choose a record set with data for analysis
if not dataframes:
    print("No record sets loaded for analysis.")
else:
    # For demonstration, select the first loaded record set
    selected_record_set_id = next(iter(dataframes))
    df = dataframes[selected_record_set_id]

    print(f"Performing EDA on record set: {selected_record_set_id}")
    print("Available columns (referenced by @id):")
    print(df.columns.tolist())

    # Attempt to select a numeric field by type (first float/int field by @id, if detected)
    import numpy as np
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id is None:
        print("No numeric field detected for demonstration.")
    else:
        # Example: filter, normalize, and group
        # Set a threshold: use mean for demonstration
        threshold = float(df[numeric_field_id].mean())
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (using mean as threshold):")
        display(filtered_df.head(3))

        # Normalize the numeric field
        norm_field = f"{numeric_field_id}_normalized"
        filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_field]].head())

        # Try grouping by another (non-numeric) field
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable non-numeric field found for grouping.")

## 5. Visualization

Visualize distributions or relationships between relevant fields. All field references must use their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if EDA provided a numeric field and DataFrame
if "filtered_df" in locals() and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field, visualize group means
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- This notebook demonstrated how to load, explore, and process a Croissant-described dataset using `mlcroissant`.
- All entities (record sets, fields, columns) were referenced by their `@id` to ensure unambiguous field usage and reproducibility.
- Further analysis can leverage the full data structure and rich metadata exposed via the Croissant schema.